# Sprint 2 - RAG, Hybrid Retrieval, Reranking, and HyDE

This notebook builds a small retrieval pipeline on top of the shared helper core. Students will index lesson notes into ChromaDB, compare semantic and keyword matching, blend them with hybrid search, rerank candidates, and rewrite a query with HyDE.


## 1. Install the helper core from GitHub

Run this first in Colab. It clones or updates the current GitHub source, installs it editable, and adds the cloned `src/` directory to this kernel before any helper imports.


In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

REPO_URL = "https://github.com/richhiey/ai-app-dev_Mod-A.git"
REPO_BRANCH = "main"
WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORK_DIR / "ai-app-dev_Mod-A"
SRC_DIR = REPO_DIR / "src"


def run(command: list[str]) -> None:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Rename it or remove it, then rerun this cell.")
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

if not SRC_DIR.exists():
    raise RuntimeError(f"Expected helper source directory not found: {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()

print(f"Using helper core source at: {SRC_DIR}")


## 2. Add your OpenRouter key and imports

Embeddings, reranking, and HyDE all route through enabled OpenRouter models. Store `OPENROUTER_API_KEY` in Colab Secrets when possible; the cell below falls back to a hidden prompt.


In [ ]:
import os
from getpass import getpass


def load_openrouter_key() -> str:
    key = os.getenv("OPENROUTER_API_KEY")
    if key:
        return key

    try:
        from google.colab import userdata

        key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        key = None

    if not key:
        key = getpass("OpenRouter API key: ")
    if not key:
        raise RuntimeError("OPENROUTER_API_KEY is required for this notebook's model calls.")

    os.environ["OPENROUTER_API_KEY"] = key
    return key


_ = load_openrouter_key()
print("OpenRouter API key loaded.")


In [ ]:
import shutil
from pathlib import Path

from documents import chunk_text
from hybrid import HybridRetriever
from hyde import HyDERewriter
from keyword_search import BM25Retriever
from openrouter import OpenRouterClient, OpenRouterEmbedder
from rerank import OpenRouterReranker
from vector_store import ChromaStore

client = OpenRouterClient(app_title="ai-app-dev-module-a-sprint-2")


## 3. Create a tiny course corpus

In the real notebooks this can be replaced with lesson markdown, transcripts, PDFs converted to text, or project docs.


In [ ]:
COURSE_NOTES = {
    "sprint_1_structured_outputs": """
    Sprint 1 introduces the AI app scaffold. Students locate the model access layer,
    send direct chat messages through OpenRouter, and compare free-form text with
    validated JSON. Structured output is useful when a model result must drive a
    route, a UI component, a retrieval decision, or a tool call.
    """,
    "sprint_2_rag_failure_modes": """
    Naive RAG often fails because chunks are too large, chunks are missing useful
    context, the retriever returns semantically related but irrelevant passages,
    or the prompt receives too many low-quality candidates. Evaluation should look
    at the retrieved evidence before judging the final answer.
    """,
    "sprint_2_hybrid_reranking": """
    Hybrid retrieval combines semantic vector search with lexical keyword matching.
    BM25 helps when exact terms like MCP, ChromaDB, HyDE, schema, or reranking are
    important. A reranker reads the candidate passages with the query and promotes
    the passages that are most useful for the answer.
    """,
    "sprint_2_hyde": """
    HyDE means hypothetical document embeddings. The model writes a short imagined
    answer document for the query. That document is embedded and used for semantic
    retrieval, while a rewritten keyword query can still feed BM25.
    """,
    "sprint_3_tools_mcp": """
    Tool calling lets the model ask the application to run a function. Tool schemas
    describe the input contract. Robust apps handle malformed JSON arguments,
    missing required fields, unknown tools, timeouts, and retryable failures. MCP
    standardizes how tools and resources are exposed to model clients.
    """,
}

documents = []
for source_id, text in COURSE_NOTES.items():
    documents.extend(
        chunk_text(
            text,
            source_id=source_id,
            chunk_size=500,
            overlap=80,
            metadata={"source": source_id},
        )
    )

print(f"Created {len(documents)} chunks")
print(documents[0].model_dump())


## 4. Index chunks into ChromaDB

The store persists locally inside the notebook runtime. Rebuilding it keeps the lab deterministic.


In [ ]:
DB_PATH = Path("/content/module_a_chroma") if Path("/content").exists() else Path(".chroma/module_a")
shutil.rmtree(DB_PATH, ignore_errors=True)

embedder = OpenRouterEmbedder(client)
store = ChromaStore(path=DB_PATH, collection_name="module_a_lessons", embedder=embedder)
indexed = store.index(documents, batch_size=8)
print(f"Indexed {indexed} chunks into {DB_PATH}")


In [ ]:
def show_results(results, score_attr):
    for rank, result in enumerate(results, start=1):
        score = getattr(result, score_attr, None)
        source = result.document.metadata.get("source", "unknown")
        preview = result.document.text.replace("\n", " ")[:180]
        print(f"{rank}. {source} | {score_attr}={score}")
        print(f"   {preview}...\n")

query = "Why does naive RAG retrieve generic chunks, and how do rerankers help?"


## 5. Semantic search

Semantic search embeds the query and finds nearby document embeddings.


In [ ]:
semantic_results = store.semantic_search(query, top_k=3)
show_results(semantic_results, "semantic_score")


## 6. Keyword search with BM25

BM25 is a lexical baseline. It can rescue exact course vocabulary that vector search may soften.


In [ ]:
keyword = BM25Retriever.from_documents(store.all_documents())
keyword_results = keyword.search(query, top_k=3)
show_results(keyword_results, "keyword_score")


## 7. Hybrid retrieval

Hybrid retrieval blends normalized semantic and keyword scores. Tune `alpha`: closer to 1 favors semantic search, closer to 0 favors keywords.


In [ ]:
hybrid = HybridRetriever(vector_store=store, keyword_retriever=keyword, alpha=0.65)
hybrid_candidates = hybrid.search(query, top_k=5, semantic_k=8, keyword_k=8)
show_results(hybrid_candidates, "hybrid_score")


## 8. Rerank the retrieved candidates

Reranking is a second pass over the short candidate list. It is usually cheaper than asking the LLM to read the whole corpus.


In [ ]:
reranker = OpenRouterReranker(client)
reranked = reranker.rerank(query, hybrid_candidates, top_n=3)
show_results(reranked, "rerank_score")


## 9. Rewrite the query with HyDE

HyDE creates a hypothetical answer document for semantic retrieval and a keyword-rich query for BM25.


In [ ]:
hyde = HyDERewriter(client).rewrite(
    query,
    context_hint="Module A covers app scaffolds, structured output, RAG, hybrid search, reranking, tools, and MCP.",
)
print(hyde.model_dump_json(indent=2))

hyde_candidates = hybrid.search(
    query,
    top_k=5,
    semantic_k=8,
    keyword_k=8,
    semantic_query=hyde.hypothetical_document,
    keyword_query=hyde.rewritten_query,
)
show_results(hyde_candidates, "hybrid_score")


## Checkpoint

Students should compare the top chunks at each stage and explain which retrieval signal changed the ranking.
